# FloodDuo — fine-tune on your own region

The release model generalizes well, but if you have even a handful of
labeled tiles (image + mask) for a specific region, or flood type it
hasn't seen before, fine-tuning the adapters can adapt it further. This
notebook: downloads the release checkpoint, fine-tunes on a small labeled
set from one site, and checks whether that adaptation holds up on a
*separate* set of tiles from the **same region** (not the same tiles) —
using the exact same inference path as `notebook/inference_demo.ipynb`.

Only FloodDuo's adapters/fusion/decoder update (~65M params); both frozen
backbones (DINOv3-L, Clay v1.5) are untouched, so this is fast even without a
large GPU — a few minutes for a dozen tiles.

**This notebook ships with a small example set (12 FloodPlanet tiles for
fine-tuning + 15 separate tiles from the same region for evaluation) purely
to show the workflow end-to-end.** Because that example region is already
part of the release model's training data, the before/after gap you'll see
here is modest — on a genuinely new region, or flood type the model
has never seen, expect a substantially larger improvement. Swap in your own
two folders (Section 3) to fine-tune for your actual site.

## 1. Install dependencies

In [ ]:
%pip install -q torch torchvision timm rasterio einops huggingface_hub matplotlib numpy

## 2. Download the checkpoint and the example fine-tuning set

The example set is laid out as two folders, each with `image/` and `mask/`
subfolders of identically-named GeoTIFFs — `finetune/` (tiles to adapt on)
and `test/` (separate tiles from the same region, to check the adaptation
generalizes). This is the same layout to use for your own data in Section 3.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, for `import src`

from huggingface_hub import hf_hub_download, snapshot_download

REPO = "Sk-21/FloodDuo"
ckpt_path = hf_hub_download(REPO, "floodduo_full.pt")

example_root = snapshot_download(REPO, allow_patterns=["examples/finetune/*/*/*.tif"])
example_root = os.path.join(example_root, "examples", "finetune")
print("downloaded example set to:", example_root)

## 3. Point this at your own two folders

`finetune_dir`: tiles to fine-tune on for your site (a dozen is enough to
start). `test_dir`: a **separate** set of tiles from the **same region**,
not used in fine-tuning — this is what tells you whether the model actually
adapted to your site, rather than just memorizing the handful of fine-tuning
tiles. Both default to the bundled example above; replace with your own two
folders (same `image/` + `mask/` layout) to fine-tune for your actual region.

In [ ]:
from src.finetune import list_tile_folder

finetune_dir = os.path.join(example_root, "train")   # <- your fine-tuning folder
test_dir = os.path.join(example_root, "test")         # <- your separate same-region test folder

train_images, train_masks = list_tile_folder(finetune_dir)
test_images, test_masks = list_tile_folder(test_dir)
print(f"fine-tuning on {len(train_images)} tiles, evaluating on {len(test_images)} "
      f"separate tiles from the same region")

## 4. Load the release model and evaluate it *before* fine-tuning

Per-tile IoU on the test folder, as a baseline for this region. `to_reflectance`
runs automatically on each tile (see `notebook/inference_demo.ipynb` Section 4
for what that does and why).

In [ ]:
import numpy as np
import torch
from src.inference import load_model, read_image, to_reflectance, predict_array

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(ckpt_path, device=device)


def evaluate(model, image_paths, mask_paths):
    ious = []
    for ip, mp in zip(image_paths, mask_paths):
        raw, _ = read_image(ip)
        image = to_reflectance(raw, verbose=False)
        _, pred = predict_array(model, image, device=device, auto_reflectance=False)
        gt_raw, _ = read_image(mp)
        gt = gt_raw[0] > 0.5
        inter = (pred & gt).sum(); union = (pred | gt).sum()
        ious.append(float(inter / union) if union > 0 else 1.0)
    return np.array(ious)


before = evaluate(model, test_images, test_masks)
print(f"BEFORE fine-tuning: mean IoU {before.mean():.3f}  (n={len(before)} test tiles)")

## 5. Fine-tune

Trains on random 512px crops of the fine-tuning folder's tiles (flips/rotations
for a bit of augmentation from so few images). `epochs`/`lr` below are
reasonable defaults for fine-tuning a handful of tiles starting from a
converged checkpoint — lower than training from scratch, since the model
mostly needs a nudge, not re-learning.

In [ ]:
from src.finetune import load_tiles, finetune

images, masks = load_tiles(train_images, train_masks, verbose=False)
history = finetune(model, images, masks, epochs=40, lr=1e-4, device=device, log_every=10)

## 6. Evaluate *after* fine-tuning, on the same region

Same test folder as Section 4 — different tiles from the ones just fine-tuned
on, so this measures whether the adaptation generalized across the region,
not whether the model memorized the fine-tuning tiles.

In [ ]:
after = evaluate(model, test_images, test_masks)
print(f"AFTER fine-tuning:  mean IoU {after.mean():.3f}  (n={len(after)} test tiles)")
print(f"delta: {after.mean() - before.mean():+.3f}  "
      f"({int((after > before).sum())} improved, {int((after < before).sum())} worsened, "
      f"{int((after == before).sum())} unchanged, out of {len(before)})")

## 7. Visualize one example from the test folder

In [ ]:
import rasterio
import matplotlib.pyplot as plt


def false_color(img):
    x = img[[3, 2, 1]].astype(float)
    out = np.empty_like(x)
    for k in range(3):
        lo, hi = np.percentile(x[k], 2), np.percentile(x[k], 98)
        out[k] = np.clip((x[k] - lo) / (hi - lo + 1e-6), 0, 1)
    return np.transpose(out, (1, 2, 0))


i = int(np.argmax(after - before))  # the test tile fine-tuning helped most, for illustration
raw, _ = read_image(test_images[i])
image = to_reflectance(raw, verbose=False)
with rasterio.open(test_masks[i]) as src:
    gt = src.read(1) > 0.5
_, pred = predict_array(model, image, device=device, auto_reflectance=False)

fig, ax = plt.subplots(1, 3, figsize=(13, 5))
ax[0].imshow(false_color(raw)); ax[0].set_title("False color")
ax[1].imshow(gt, cmap="Blues", vmin=0, vmax=1); ax[1].set_title("Ground truth")
ax[2].imshow(pred, cmap="Blues", vmin=0, vmax=1)
ax[2].set_title(f"Fine-tuned prediction (IoU {after[i]:.2f}, was {before[i]:.2f})")
for a in ax: a.set_xticks([]); a.set_yticks([])
fig.tight_layout()

## 8. Save the fine-tuned weights

In [ ]:
out_path = "floodduo_finetuned.pt"
torch.save(model.state_dict(), out_path)
print("saved:", out_path)
# Reload later with: model.load_state_dict(torch.load(out_path))